## Tổng quan hệ thống

Dự án dùng dữ liệu bán hàng theo ngày để dự báo sales trong 16 ngày tới cho từng cặp (store_nbr, family). Luồng chính gồm 4 bước:
1. Data loading và reindexing
2. Feature engineering
3. Model training và prediction
4. Validation bằng holdout và RMSLE

## Các file chính liên quan

- src/data.py: đọc dữ liệu và xây lại lịch đầy đủ cho từng chuỗi thời gian
- src/features.py: xây features như holiday, calendar, oil, promotion, lag
- src/models.py: baseline và mô hình deterministic
- src/validation.py: holdout split, no leakage, RMSLE

## Kế hoạch QA gồm 6 task

1. Kiểm tra tính toàn vẹn dữ liệu
2. Kiểm tra holdout split và thời gian
3. Kiểm tra baseline seasonal-naive
4. Kiểm tra deterministic model và reproducibility
5. Kiểm tra feature engineering và leakage
6. Kiểm tra metric và kết quả đánh giá

Sau khi đọc phần này, bạn có thể chạy các cell tiếp theo để xem code thực hiện từng task và kết quả của nó.

In [35]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Make the repository root importable when running the notebook from notebooks/
repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src import data, features, models, validation

EXPECTED_SERIES = 1782
EXPECTED_DAYS = 1688
EXPECTED_ROWS = EXPECTED_SERIES * EXPECTED_DAYS
EXPECTED_CLOSED_DAYS = EXPECTED_SERIES * 4


def check_data_integrity() -> pd.DataFrame:
    """Validate raw data loading and gap-free reindexing."""
    print('Task 1/6: Checking data integrity...')
    train = data.load_train()
    print(f'   - raw train shape: {train.shape}')

    gapfree = data.reindex_series_gapfree(train)
    print(f'   - gap-free train shape: {gapfree.shape}')
    validation.assert_gapfree(gapfree)
    print('   - assert_gapfree: OK')

    n_series = gapfree.groupby(['store_nbr', 'family'], observed=True).ngroups
    if n_series != EXPECTED_SERIES:
        raise AssertionError(f'unexpected number of series: {n_series} != {EXPECTED_SERIES}')
    if gapfree.shape[0] != EXPECTED_ROWS:
        raise AssertionError(f'unexpected total gap-free rows: {gapfree.shape[0]} != {EXPECTED_ROWS}')

    closed_days = int(gapfree['was_closed'].sum())
    if closed_days != EXPECTED_CLOSED_DAYS:
        raise AssertionError(f'unexpected closed-day count: {closed_days} != {EXPECTED_CLOSED_DAYS}')

    first_date = gapfree['date'].min().date()
    last_date = gapfree['date'].max().date()
    print(f'   - date range: {first_date} -> {last_date}')
    print(f'   - number of series: {n_series}')
    print(f'   - inserted closed days: {closed_days}')
    return gapfree


def check_holdout_split(gapfree: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Check time-based holdout split and leakage guard."""
    print('Task 2/6: Checking holdout split...')
    train, holdout = validation.train_holdout_split(gapfree)
    holdout_start = holdout['date'].min()
    holdout_end = holdout['date'].max()

    print(f"   - train end: {train['date'].max().date()}")
    print(f"   - holdout start: {holdout_start.date()}")
    print(f"   - holdout end: {holdout_end.date()}")
    print(f"   - holdout days: {holdout['date'].nunique()}")

    validation.assert_no_leak(train, holdout_start, strict=True)
    print('   - assert_no_leak(train, holdout_start, strict=True): OK')
    return train, holdout


def validate_predictions(preds: pd.DataFrame, model_name: str) -> None:
    """Ensure predictions are well-formed and non-negative."""
    expected_columns = ['store_nbr', 'family', 'date', 'sales_pred']
    missing_columns = [col for col in expected_columns if col not in preds.columns]
    if missing_columns:
        raise AssertionError(f'{model_name} predictions missing expected columns: {missing_columns}')

    expected_rows = EXPECTED_SERIES * validation.HORIZON_DAYS
    if len(preds) != expected_rows:
        raise AssertionError(f'{model_name} predictions row count {len(preds)} != {expected_rows}')

    if preds['sales_pred'].isna().any():
        raise AssertionError(f'{model_name} predictions contain NaN values')
    if np.isinf(preds['sales_pred'].to_numpy()).any():
        raise AssertionError(f'{model_name} predictions contain infinite values')
    if (preds['sales_pred'] < 0).any():
        raise AssertionError(f'{model_name} predictions contain negative values')

    print(f'   - {model_name} prediction sanity: {len(preds)} rows, no NaN/inf/negative')


def check_baseline(train: pd.DataFrame, holdout: pd.DataFrame) -> float:
    """Evaluate seasonal-naive baseline on the validation holdout."""
    print('Task 3/6: Checking seasonal-naive baseline...')
    preds = models.seasonal_naive_predict(train, validation.HORIZON_DAYS)
    validate_predictions(preds, 'seasonal-naive baseline')

    scored = holdout.merge(preds, on=['store_nbr', 'family', 'date'], how='left')
    if scored['sales_pred'].isna().any():
        raise AssertionError('baseline predictions missing for some holdout rows')
    score = validation.rmsle(scored['sales'], scored['sales_pred'])
    print(f'   - baseline RMSLE: {score:.5f}')
    return score


def check_deterministic(train: pd.DataFrame, holdout: pd.DataFrame) -> float:
    """Check deterministic model reproducibility and holdout score."""
    print('Task 4/6: Checking deterministic model...')
    model_a = models.DeterministicModel().fit(train)
    preds_a = model_a.predict(validation.HORIZON_DAYS)
    validate_predictions(preds_a, 'deterministic model (run 1)')

    model_b = models.DeterministicModel().fit(train)
    preds_b = model_b.predict(validation.HORIZON_DAYS)
    validate_predictions(preds_b, 'deterministic model (run 2)')

    if not preds_a.equals(preds_b):
        raise AssertionError('Deterministic model predictions differ across repeated training runs')
    print('   - deterministic reproducibility: run 1 and run 2 predictions identical')

    scored = holdout.merge(preds_a, on=['store_nbr', 'family', 'date'], how='left')
    if scored['sales_pred'].isna().any():
        raise AssertionError('deterministic model predictions missing for some holdout rows')
    score = validation.rmsle(scored['sales'], scored['sales_pred'])
    print(f'   - deterministic RMSLE: {score:.5f}')
    return score


def check_feature_engineering(gapfree: pd.DataFrame, train: pd.DataFrame) -> None:
    """Validate holiday/calendar/oil/promotion/lag feature builders."""
    print('Task 5/6: Checking feature engineering...\n   building holiday/calendar/oil/promotion/lag features')
    stores = data.load_stores()
    holidays = data.load_holidays()
    oil = data.load_oil()
    transactions = data.load_transactions()

    holiday_features = features.make_holiday_features(holidays, stores)
    if holiday_features['is_holiday'].isin([0, 1]).all() is False:
        raise AssertionError('holiday indicator values are not binary')
    print(f'   - holiday rows: {len(holiday_features)}')

    calendar_features = features.make_calendar_features(gapfree['date'].unique())
    if calendar_features['dayofweek'].isna().any():
        raise AssertionError('calendar features contain missing values')
    print(f'   - calendar rows: {len(calendar_features)}')

    oil_features = features.make_oil_features(oil, gapfree['date'].unique())
    if oil_features['oil'].isna().any():
        raise AssertionError('oil features contain missing values')
    print(f'   - oil features: {len(oil_features)} rows, no gaps')

    promo_features = features.make_promotion_features(gapfree)
    if promo_features['onpromotion'].isna().any():
        raise AssertionError('promotion features contain missing values')
    print(f'   - promotion rows: {len(promo_features)}')

    lag_features = features.make_lag_features(gapfree, transactions)
    holdout_start = train['date'].max() + pd.Timedelta(days=1)
    validation.assert_no_leak(
        lag_features.loc[lag_features['date'] < holdout_start],
        holdout_start,
        strict=True,
        base_lag=validation.HORIZON_DAYS,
    )
    print('   - lag feature names and base_lag guard OK')
    print('   - feature engineering checks passed')


def check_linear_features(gapfree: pd.DataFrame, train: pd.DataFrame, holdout: pd.DataFrame, baseline_score: float, deterministic_score: float) -> float:
    """Build feature matrix and train LinearFeatureModel to validate features add value."""
    print('Task 5b/6: Training linear model on features...')
    
    KEY = ['store_nbr', 'family']
    dates = pd.DatetimeIndex(np.sort(gapfree['date'].unique()))
    stores = data.load_stores()
    holidays = data.load_holidays()
    oil = data.load_oil()
    transactions = data.load_transactions()
    
    # Build each feature group
    det  = features.make_deterministic_features(dates).rename_axis('date').reset_index()
    cal  = features.make_calendar_features(dates).rename_axis('date').reset_index()
    oil_feat = features.make_oil_features(oil, dates).rename_axis('date').reset_index()
    hol  = features.make_holiday_features(holidays, stores)
    lags = features.make_lag_features(gapfree, transactions)
    
    DET_COLS  = list(det.columns[1:])
    CAL_COLS  = list(cal.columns[1:])
    HOL_COLS  = ['is_holiday', 'is_work_day', 'is_national_holiday']
    PROMO_OIL = ['onpromotion', 'oil']
    LAG_COLS  = [c for c in lags.columns if c not in KEY + ['date']]
    
    # Compose feature matrix
    matrix = (
        gapfree[KEY + ['date', 'sales', 'onpromotion']]
        .merge(det, on='date')
        .merge(cal, on='date')
        .merge(oil_feat, on='date')
        .merge(hol, on=['store_nbr', 'date'], how='left')
    )
    matrix[HOL_COLS] = matrix[HOL_COLS].fillna(0).astype('int8')
    matrix = matrix.merge(lags, on=KEY + ['date'], how='left')
    
    # Split train/holdout by date
    holdout_start = train['date'].max() + pd.Timedelta(days=1)
    train_m = matrix[matrix['date'] <= train['date'].max()].copy()
    hold_m  = matrix[matrix['date'] >= holdout_start].copy()
    
    # Check no leakage on fitted features
    all_cols = DET_COLS + HOL_COLS + CAL_COLS + PROMO_OIL + LAG_COLS
    validation.assert_no_leak(
        train_m[KEY + ['date'] + all_cols],
        holdout_start,
        strict=True,
        base_lag=validation.HORIZON_DAYS,
    )
    print(f'   - no-leak check passed for {len(all_cols)} features')
    
    # Train LinearFeatureModel
    model = models.LinearFeatureModel(all_cols).fit(train_m)
    preds = model.predict(hold_m)
    
    # Score
    truth = holdout[KEY + ['date', 'sales']].copy()
    scored = truth.merge(preds, on=KEY + ['date'])
    if scored['sales_pred'].isna().any():
        raise AssertionError('linear model predictions missing for some holdout rows')
    
    score = validation.rmsle(scored['sales'], scored['sales_pred'])
    print(f'   - linear model RMSLE: {score:.5f}')
    print(f'   - vs baseline:      {score - baseline_score:+.5f}')
    print(f'   - vs deterministic: {score - deterministic_score:+.5f}')
    return score


def main() -> None:
    """Run the full QA workflow end to end."""
    root = repo_root
    print(f'QA script running from: {root}\n')
    gapfree = check_data_integrity()
    train, holdout = check_holdout_split(gapfree)
    baseline_score = check_baseline(train, holdout)
    deterministic_score = check_deterministic(train, holdout)
    check_feature_engineering(gapfree, train)

    print('\nSummary:')
    print(f' - baseline RMSLE:      {baseline_score:.5f}')
    print(f' - deterministic RMSLE: {deterministic_score:.5f}')
    print('\nQA checks completed successfully.')


## Task 1: Kiểm tra tính toàn vẹn dữ liệu

Mục tiêu của task này là xác nhận dữ liệu đầu vào đủ đầy, có thể reindex thành lịch ngày đầy đủ, và các ngày đóng cửa được chèn đúng với sales = 0 và was_closed = True.

Code liên quan:
- src/data.py: load_train(), reindex_series_gapfree()
- src/validation.py: assert_gapfree()

Mục đích review:
- xác nhận số series là 1,782
- xác nhận số dòng sau reindex là `1,782 × 1,688`
- xác nhận các ngày đóng cửa được chèn với `sales = 0`

Khi chạy cell bên dưới, notebook sẽ gọi hàm check_data_integrity() để kiểm tra toàn bộ dữ liệu và in ra các số liệu quan trọng.

In [26]:
gapfree = check_data_integrity()

Task 1/6: Checking data integrity...
   - raw train shape: (3000888, 6)
   - gap-free train shape: (3008016, 7)
   - assert_gapfree: OK
   - date range: 2013-01-01 -> 2017-08-15
   - number of series: 1782
   - inserted closed days: 7128


Nếu ô trên chạy thành công, nghĩa là dữ liệu đã được reindex đúng và không có lỗ ngày nào.

## Task 2: Kiểm tra holdout split và thời gian

Task này kiểm tra xem cách tách train/holdout có đúng theo thời gian hay không, đồng thời đảm bảo không có leakage từ dữ liệu tương lai vào train.

Code liên quan:
- src/validation.py: train_holdout_split(), assert_no_leak()

Mục đích review:
- holdout phải bắt đầu ngay sau train
- train không được chứa dữ liệu từ holdout
- holdout có đúng 16 ngày

Nếu cell chạy thành công, nghĩa là holdout bắt đầu đúng ngay sau train và không có dữ liệu từ tương lai bị dùng sai.

In [27]:
train, holdout = check_holdout_split(gapfree)

Task 2/6: Checking holdout split...
   - train end: 2017-07-30
   - holdout start: 2017-07-31
   - holdout end: 2017-08-15
   - holdout days: 16
   - assert_no_leak(train, holdout_start, strict=True): OK


## Task 3: Kiểm tra baseline seasonal-naive

Baseline là điểm tham chiếu đầu tiên. Task này đánh giá mô hình seasonal-naive trên holdout và kiểm tra xem output predictions có đúng schema và hợp lệ không.

Code liên quan:
- src/models.py: seasonal_naive_predict()
- src/validation.py: rmsle()

Mục đích review:
- predictions có đủ cột đầu ra
- số dòng đúng bằng `1,782 × 16`
- không có NaN / inf / giá trị âm

Kết quả chạy cell này sẽ cho bạn thấy RMSLE của baseline trên holdout.

In [28]:
baseline_score = check_baseline(train, holdout)

Task 3/6: Checking seasonal-naive baseline...


   - seasonal-naive baseline prediction sanity: 28512 rows, no NaN/inf/negative
   - baseline RMSLE: 0.61704


Baseline này là điểm tham chiếu để so sánh với các mô hình khác.

## Task 4: Kiểm tra deterministic model và reproducibility

Task này chạy mô hình deterministic hai lần để xác nhận rằng cùng một dữ liệu đầu vào cho ra kết quả giống nhau. Đây là kiểm tra quan trọng cho tính ổn định và dễ review.

Code liên quan:
- src/models.py: DeterministicModel.fit(), DeterministicModel.predict()
- src/validation.py: rmsle()

Mục đích review:
- predictions có đủ cột đầu ra
- số dòng đúng bằng `1,782 × 16`
- không có NaN / inf / giá trị âm
- các lần chạy lặp lại cho kết quả giống nhau

Nếu hai lần chạy cho kết quả giống nhau, model được coi là ổn định.

In [29]:
deterministic_score = check_deterministic(train, holdout)

Task 4/6: Checking deterministic model...
   - deterministic model (run 1) prediction sanity: 28512 rows, no NaN/inf/negative
   - deterministic model (run 2) prediction sanity: 28512 rows, no NaN/inf/negative
   - deterministic reproducibility: run 1 and run 2 predictions identical
   - deterministic RMSLE: 0.62188


## Task 5: Kiểm tra feature engineering và training linear model

Task này kiểm tra các feature như holiday, calendar, oil, promotion và lag có hợp lệ và không bị leak không. Sau đó xây dựng feature matrix hoàn chỉnh, train LinearFeatureModel và đánh giá trên holdout.

Code liên quan:
- src/features.py: make_holiday_features(), make_calendar_features(), make_oil_features(), make_promotion_features(), make_lag_features()
- src/models.py: LinearFeatureModel.fit(), LinearFeatureModel.predict()
- src/validation.py: assert_no_leak(), rmsle()

Mục đích review:
- holiday feature binary đúng
- calendar feature không thiếu giá trị
- oil feature không còn gap sau fill
- promotion feature không có NaN
- lag feature không sử dụng dữ liệu tương lai
- linear model trên features có improve vs baseline hay không
- linear model score so sánh vs deterministic

Nếu task này chạy không lỗi, có nghĩa là feature engineering đã đúng, không bị leak, và linear model training thành công.

In [30]:
check_feature_engineering(gapfree, train)

Task 5/6: Checking feature engineering...
   building holiday/calendar/oil/promotion/lag features
   - holiday rows: 9139
   - calendar rows: 1688
   - oil features: 1688 rows, no gaps
   - promotion rows: 3008016
   - lag feature names and base_lag guard OK
   - feature engineering checks passed


In [36]:
linear_score = check_linear_features(gapfree, train, holdout, baseline_score, deterministic_score)

Task 5b/6: Training linear model on features...
   - feature names asserted for 33 columns
   - no-leak check passed for 33 features
   - linear model RMSLE: 0.50991
   - vs baseline:      -0.10713
   - vs deterministic: -0.11197


Nếu phần này không báo lỗi, có nghĩa là feature engineering đã hợp lệ và không bị rò leak.

## Task 6: Kiểm tra metric, kết quả đánh giá và so sánh các mô hình

Task này tổng hợp lại các kết quả từ toàn bộ QA pipeline. Reviewer cần xem xét các metric đã dùng và xác nhận các model được đánh giá đúng trên cùng một holdout.

Kết quả từ các task trước:
- **Task 1:** Dữ liệu được reindex đúng, 1,782 series, 1,688 days, 7,128 closed days
- **Task 2:** Holdout split chính xác, 16 ngày, không leak
- **Task 3:** Baseline seasonal-naive đã evaluate
- **Task 4:** Deterministic model reproducible, score ổn định
- **Task 5:** Feature engineering đúng, không leak, linear model đã train

Mục đích review:
- Dùng cùng một metric RMSLE cho mọi mô hình (kiểm tra thủ công code RMSLE xem đúng chưa và cách gọi lại hàm trong các model)
- So sánh baseline, deterministic, và linear model trên cùng holdout 16 ngày (xem ở output cell task 5b)
- Xác nhận linear model có improve vs baseline không (có)
